# Political Polarity Text Analyzer

**Google Colab Notebook** - Analyze the political polarity of text documents.

This notebook provides a complete, self-contained pipeline for:
1. Training a polarity classification model on labeled text
2. Analyzing new text documents for political lean
3. Extracting discriminative phrases

---

### Ethics Notice

**This system classifies TEXT CONTENT, not individuals.**
- Do NOT use outputs to infer any individual's political beliefs
- Do NOT use to target or discriminate against any group
- Outlet-level labels are weak supervision and may not match article-level ideology
- See the Model Card section at the bottom for full ethical guidance

---

## 1. Setup & Installation

In [ ]:
# Install dependencies (run once)
!pip install -q numpy==1.26.4 pandas==2.2.1 scikit-learn==1.4.1.post1 \
    scipy==1.12.0 matplotlib==3.8.3 pyyaml==6.0.1 tqdm==4.66.2

# Optional: install torch + transformers for the stronger model
# !pip install -q torch==2.2.1 transformers==4.38.2 accelerate==0.27.2

print("Setup complete!")

In [ ]:
import json
import random
import hashlib
import re
import unicodedata
import logging
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Labels
CLASS_NAMES = ["left", "center", "right"]
CLASS_TO_INT = {"left": 0, "center": 1, "right": 2}
INT_TO_CLASS = {0: "left", 1: "center", 2: "right"}

print("Imports ready.")

## 2. Text Preprocessing Functions

In [ ]:
def normalize_text(text: str) -> str:
    """Clean and normalize text."""
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\u2018", "'").replace("\u2019", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2014", " -- ").replace("\u2013", " - ")
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"\S+@\S+\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def compute_text_hash(text: str) -> str:
    """Hash for deduplication."""
    normalized = re.sub(r"\s+", " ", text.lower().strip())
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


def deduplicate(df: pd.DataFrame) -> pd.DataFrame:
    """Remove duplicate texts."""
    df = df.copy()
    df["_hash"] = df["text"].apply(compute_text_hash)
    df = df.drop_duplicates(subset="_hash", keep="first")
    return df.drop(columns=["_hash"]).reset_index(drop=True)


print("Preprocessing functions ready.")

## 3. Load Your Data

### Option A: Use Synthetic Data (for testing)

In [ ]:
# === SYNTHETIC DATA (for testing only) ===
# Replace this section with your own articles

LEFT_TEXTS = [
    "The government should increase funding for public healthcare programs to ensure universal coverage for all citizens.",
    "Climate change is the defining challenge of our generation. We need aggressive action to reduce carbon emissions.",
    "Income inequality has reached alarming levels. Progressive taxation and raising the minimum wage would help.",
    "Gun violence is a public health crisis. Common-sense gun safety legislation is urgently needed.",
    "Investments in public education, including universal pre-K and affordable college, are crucial for our future.",
    "Workers deserve stronger protections and the right to organize unions.",
    "Immigration reform should provide a pathway to citizenship for undocumented immigrants.",
    "Social safety net programs like SNAP are vital investments that help families.",
    "Criminal justice reform must address systemic inequities and end mass incarceration.",
    "Reproductive rights are fundamental healthcare decisions that should be protected.",
]

RIGHT_TEXTS = [
    "Lower taxes and reduced government regulation are the keys to economic growth.",
    "The Second Amendment clearly protects the individual right to keep and bear arms.",
    "Securing our borders is a matter of national sovereignty and security.",
    "Free market solutions, not government mandates, drive innovation in healthcare.",
    "Traditional family values and religious liberty are the foundation of a strong society.",
    "A strong national defense is essential for protecting American interests abroad.",
    "School choice empowers parents to select the best education for their children.",
    "The national debt threatens future generations. Fiscal discipline is needed.",
    "Energy independence through domestic oil and gas production creates jobs.",
    "Law enforcement officers deserve our support. Defunding police makes communities less safe.",
]

CENTER_TEXTS = [
    "The debate over healthcare policy involves legitimate perspectives on both sides.",
    "Environmental policy must balance ecological protection with economic realities.",
    "The federal budget reflects competing priorities that all have merit.",
    "Education reform efforts have produced mixed results across different states.",
    "Trade policy involves complex tradeoffs between prices, jobs, and relationships.",
    "Immigration policy discussions often generate more heat than light.",
    "Government regulation and economic growth have a complex relationship.",
    "Technology policy raises questions that don't map onto traditional political lines.",
    "Infrastructure investment enjoys bipartisan support in principle.",
    "Addressing the opioid crisis requires cooperation across party lines.",
]

# Build dataset
texts = LEFT_TEXTS * 5 + CENTER_TEXTS * 5 + RIGHT_TEXTS * 5
labels = ["left"] * 50 + ["center"] * 50 + ["right"] * 50

df = pd.DataFrame({"text": texts, "label": labels})
df["text"] = df["text"].apply(normalize_text)
df = deduplicate(df)

print(f"Dataset: {len(df)} documents")
print(df["label"].value_counts())

### Option B: Paste Your Own Articles

Uncomment and edit the cell below to add your own labeled articles.

In [ ]:
# === PASTE YOUR OWN ARTICLES HERE ===
# Uncomment and add your texts with labels

# my_articles = [
#     {"text": "Paste article 1 text here...", "label": "left"},
#     {"text": "Paste article 2 text here...", "label": "right"},
#     {"text": "Paste article 3 text here...", "label": "center"},
#     # Add more articles...
# ]
# df = pd.DataFrame(my_articles)
# df["text"] = df["text"].apply(normalize_text)
# df = deduplicate(df)
# print(f"Your dataset: {len(df)} documents")
# print(df["label"].value_counts())

## 4. Train/Test Split

In [ ]:
# Encode labels
df["label_int"] = df["label"].map(CLASS_TO_INT)

# Split
train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df["label_int"], random_state=SEED
)

print(f"Train: {len(train_df)}, Test: {len(test_df)}")
print("\nTrain distribution:")
print(train_df["label"].value_counts())

## 5. Build TF-IDF Features & Train Model

In [ ]:
# TF-IDF vectorization
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 3),
    min_df=1,
    max_df=0.95,
    sublinear_tf=True,
    stop_words="english",
)

X_train = vectorizer.fit_transform(train_df["text"])
X_test = vectorizer.transform(test_df["text"])
y_train = train_df["label_int"].values
y_test = test_df["label_int"].values

print(f"Features: {X_train.shape[1]}")

# Train Logistic Regression
model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver="lbfgs",
    class_weight="balanced",
    random_state=SEED,
)
model.fit(X_train, y_train)

print("Model trained!")

## 6. Evaluate

In [ ]:
# Predictions
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

# Metrics
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy: {acc:.4f}")
print(f"Macro F1: {f1:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

# Confusion Matrix
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, cmap="Blues")
ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.show()

## 7. Discriminative Phrases per Class

In [ ]:
feature_names = vectorizer.get_feature_names_out()

print("=" * 60)
print("TOP PHRASES DRIVING EACH CLASS")
print("=" * 60)

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    coefs = model.coef_[cls_idx]
    top_idx = np.argsort(coefs)[::-1][:15]
    
    print(f"\n--- {cls_name.upper()} ---")
    for i, idx in enumerate(top_idx):
        print(f"  {i+1:2d}. {feature_names[idx]:30s} (coef: {coefs[idx]:+.4f})")

# Chi-square discriminative phrases
print("\n" + "=" * 60)
print("CHI-SQUARE DISCRIMINATIVE PHRASES")
print("=" * 60)

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    y_binary = (y_train == cls_idx).astype(int)
    chi2_scores, _ = chi2(X_train, y_binary)
    top_idx = np.argsort(chi2_scores)[::-1][:10]
    
    print(f"\n--- {cls_name.upper()} ---")
    for i, idx in enumerate(top_idx):
        print(f"  {i+1:2d}. {feature_names[idx]:30s} (chi2: {chi2_scores[idx]:.2f})")

## 8. Analyze New Text

Paste any text below to get its polarity rating.

In [ ]:
def analyze_text(text: str) -> dict:
    """Analyze a single text for polarity."""
    clean = normalize_text(text)
    X = vectorizer.transform([clean])
    proba = model.predict_proba(X)[0]
    pred_idx = int(np.argmax(proba))
    polarity = float(proba[2] - proba[0])  # P(right) - P(left)
    
    # Top features in this text
    nonzero = X.nonzero()[1]
    coefs = model.coef_[pred_idx]
    feature_scores = [
        (feature_names[j], float(coefs[j] * X[0, j]))
        for j in nonzero
    ]
    feature_scores.sort(key=lambda x: abs(x[1]), reverse=True)
    
    return {
        "predicted_label": CLASS_NAMES[pred_idx],
        "probabilities": {cn: float(proba[i]) for i, cn in enumerate(CLASS_NAMES)},
        "polarity_score": polarity,
        "top_features": feature_scores[:10],
    }


def display_result(result: dict):
    """Pretty-print analysis results."""
    print("\n" + "=" * 50)
    print(f"PREDICTED: {result['predicted_label'].upper()}")
    print(f"Polarity:  {result['polarity_score']:+.3f} (Left -1 <-> +1 Right)")
    print("\nProbabilities:")
    for cls, prob in result["probabilities"].items():
        bar = chr(9608) * int(prob * 30) + chr(9617) * (30 - int(prob * 30))
        print(f"  {cls:>8}: {prob:.3f} {bar}")
    
    print("\nTop contributing features:")
    for feat, score in result["top_features"]:
        direction = "->R" if score > 0 else "<-L"
        print(f"  {feat:30s} {score:+.4f} {direction}")
    
    print("\n" + chr(9888) + " This classifies TEXT CONTENT, not the author.")
    print("=" * 50)

In [ ]:
# === PASTE YOUR TEXT HERE TO ANALYZE ===

sample_text = """
The government should invest more in renewable energy and public 
healthcare programs to ensure that all citizens have access to 
affordable care and a clean environment.
"""

result = analyze_text(sample_text)
display_result(result)

In [ ]:
# Try another text

sample_text_2 = """
Lower taxes and reduced regulation would stimulate economic growth 
and create more jobs. The free market is the best mechanism for 
driving innovation and prosperity.
"""

result2 = analyze_text(sample_text_2)
display_result(result2)

In [ ]:
# Try a center/balanced text

sample_text_3 = """
The debate over this policy involves legitimate perspectives on 
both sides. While some advocate for government intervention, others 
prefer market-based approaches. The evidence is mixed.
"""

result3 = analyze_text(sample_text_3)
display_result(result3)

## 9. Batch Analysis

Analyze multiple texts at once.

In [ ]:
# Batch analysis example
batch_texts = [
    "Universal healthcare is a human right that the government must provide.",
    "The Second Amendment is sacred and shall not be infringed.",
    "Both parties need to work together to solve infrastructure problems.",
    "Workers need stronger union protections and higher minimum wages.",
    "Tax cuts for businesses create jobs and grow the economy.",
]

batch_results = [analyze_text(t) for t in batch_texts]

print(f"{'Text (first 60 chars)':60s} {'Pred':>8s} {'Polarity':>10s}")
print("-" * 80)
for text, result in zip(batch_texts, batch_results):
    snippet = text[:57] + "..." if len(text) > 60 else text
    print(f"{snippet:60s} {result['predicted_label']:>8s} {result['polarity_score']:+10.3f}")

# Polarity distribution
scores = [r["polarity_score"] for r in batch_results]
print(f"\nMean polarity: {np.mean(scores):+.3f}")
print(f"Std polarity:  {np.std(scores):.3f}")

## 10. Polarity Distribution Plot

In [ ]:
# Score all test documents
test_proba = model.predict_proba(X_test)
test_polarity = test_proba[:, 2] - test_proba[:, 0]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(test_polarity, bins=30, range=(-1, 1), color="#4a86c8", 
        alpha=0.7, edgecolor="black")
ax.axvline(0, color="gray", linestyle="--", alpha=0.5)
ax.axvline(np.mean(test_polarity), color="red", linewidth=2,
           label=f"Mean: {np.mean(test_polarity):.3f}")
ax.set_xlabel("Polarity Score (Left <- -> Right)")
ax.set_ylabel("Count")
ax.set_title("Test Set Polarity Distribution")
ax.legend()
plt.tight_layout()
plt.show()

---

## Model Card

### Intended Use
- Analytical classification of text content for research purposes
- Studying media framing and political language patterns

### NOT Intended For
- Inferring any individual's political beliefs or affiliation
- Targeting, profiling, or discriminating against individuals or groups
- Automated content moderation or censorship decisions

### Limitations
- Trained on synthetic/limited data; real-world performance will differ
- Outlet-level labels are weak supervision (article != outlet ideology)
- May conflate topic with ideology (e.g., discussing healthcare != left)
- English-only; US political context
- 3-class scheme is a simplification of a complex spectrum

### Ethical Risks
- **Individual profiling**: Never use to assign political labels to people
- **Bias amplification**: Model may reflect biases in training data
- **Topic-ideology conflation**: Discussing a topic is not the same as advocating for a political position
- **Temporal drift**: Political language evolves; model may become outdated